In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt
/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/BBox_List_2017_Official_NIH.csv
/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt
/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv
/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/pretrained_model.h5


In [3]:
import os
import numpy as np
import pandas as pd
from glob import glob
from itertools import chain

def prepare_dataframe(csv_path, image_root=None):

    if image_root is None:
        image_root = os.path.dirname(csv_path)
        
    data = pd.read_csv(csv_path)

    data['Patient Age'] = data['Patient Age'].astype(str)
    data['Patient Age'] = data['Patient Age'].str.extract(r'(\d+)')
    data['Patient Age'] = pd.to_numeric(data['Patient Age'], errors='coerce')
    data = data.dropna(subset=['Patient Age'])
    data = data[data['Patient Age'] < 100]
    data['Patient Age'] = data['Patient Age'].astype(int)

    # Map image paths
    data_image_paths = {
        os.path.basename(x): x
        for x in glob(os.path.join(image_root, '**', '*.png'), recursive=True)
    }


    print('Scans found:', len(data_image_paths), ', Total Headers:', data.shape[0])

    data['path'] = data['Image Index'].map(data_image_paths.get)
    data = data.dropna(subset=['path'])

    # Clean labels
    data['Finding Labels'] = data['Finding Labels'].map(lambda x: x.replace('No Finding', ''))

    # Extract unique labels
    all_labels = np.unique(list(chain(*data['Finding Labels'].map(lambda x: x.split('|')).tolist())))
    all_labels = [x for x in all_labels if len(x) > 0]

    print(f"All Labels ({len(all_labels)}): {all_labels}")

    # Multi-label columns
    for c_label in all_labels:
        data[c_label] = data['Finding Labels'].map(lambda x: 1.0 if c_label in x else 0)

    return data, all_labels


In [4]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, label_cols, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.label_cols = label_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)

        labels = torch.tensor(row[self.label_cols].values.astype(np.float32))
        age = torch.tensor([row['Patient Age']], dtype=torch.float32)

        return img, age, labels


In [5]:
def sample_dataframe(data, sample_size=40000):
    sample_weights = data['Finding Labels'].map(
        lambda x: len(x.split('|')) if len(x) > 0 else 0
    ).values + 4e-2

    sample_weights = sample_weights / sample_weights.sum()

    if sample_size < len(data):
        data = data.sample(sample_size, weights=sample_weights)

    return data


In [7]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.05),
        shear=5,
        scale=(0.85, 1.15)
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])


In [8]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

def build_dataloaders(data, all_labels, batch_size=32):

    train_df, val_df = train_test_split(data, test_size=0.2, random_state=42)

    train_ds = ChestXrayDataset(train_df, all_labels, train_transform)
    val_ds   = ChestXrayDataset(val_df, all_labels, val_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4)

    return train_loader, val_loader


In [9]:
data, all_labels = prepare_dataframe('/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv')
data = sample_dataframe(data, sample_size=40000)
train_loader, val_loader = build_dataloaders(data, all_labels)


Scans found: 112120 , Total Headers: 112104
All Labels (14): [np.str_('Atelectasis'), np.str_('Cardiomegaly'), np.str_('Consolidation'), np.str_('Edema'), np.str_('Effusion'), np.str_('Emphysema'), np.str_('Fibrosis'), np.str_('Hernia'), np.str_('Infiltration'), np.str_('Mass'), np.str_('Nodule'), np.str_('Pleural_Thickening'), np.str_('Pneumonia'), np.str_('Pneumothorax')]


In [11]:
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [12]:
class DenseNetBackbone(nn.Module):
    def __init__(self):
        super().__init__()

        densenet = models.densenet121(weights="DEFAULT")
        self.features = densenet.features

    def forward(self, x):
        x = self.features(x)   # [B,1024,7,7]
        return x

In [13]:
class SelfAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()

        self.query = nn.Linear(input_dim, hidden_dim)
        self.key   = nn.Linear(input_dim, hidden_dim)
        self.value = nn.Linear(input_dim, hidden_dim)

        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        # x : [B, N, D]
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        attn = torch.bmm(Q, K.transpose(1,2)) / (Q.size(-1)**0.5)
        attn = self.softmax(attn)

        out = torch.bmm(attn, V)
        return out

In [14]:
class FeaturePyramidNetwork(nn.Module):
    def __init__(self, out_channels=128):
        super().__init__()

        # single-scale FPN (safe version)
        self.lateral = nn.Conv2d(1024, out_channels, kernel_size=1)

    def forward(self, x):
        x = self.lateral(x)
        return x

In [15]:
class MedFusionNet(nn.Module):
    def __init__(self, tabular_dim, num_classes):
        super().__init__()

        # Image branch
        self.backbone = DenseNetBackbone()
        self.fpn = FeaturePyramidNetwork(128)
        self.img_pool = nn.AdaptiveAvgPool2d(1)

        # Tabular branch
        self.attention = SelfAttention(tabular_dim, 64)

        # Fusion
        self.fc_fusion = nn.Linear(128 + 64, 128)

        # Classifier
        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, images, tabular):

        # ----- Image branch -----
        img_feat = self.backbone(images)
        img_feat = self.fpn(img_feat)
        img_feat = self.img_pool(img_feat).flatten(1)

        # ----- Tabular branch -----
        # reshape tabular for attention
        tabular = tabular.unsqueeze(1)   # [B,1] -> [B,1,1]

        tab_feat = self.attention(tabular).mean(dim=1)

        # ----- Fusion -----
        fused = torch.cat([img_feat, tab_feat], dim=1)
        fused = self.fc_fusion(fused)

        out = self.classifier(fused)

        return out   # NO sigmoid here

In [17]:
model = MedFusionNet(
    tabular_dim=1,        
    num_classes=len(all_labels)
).to(device)


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 162MB/s] 


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):

        # BCE with logits
        bce_loss = F.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )

        # Convert logits to probabilities
        probs = torch.sigmoid(logits)

        pt = torch.where(targets == 1, probs, 1 - probs)

        focal_weight = self.alpha * (1 - pt) ** self.gamma

        loss = focal_weight * bce_loss

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss

In [60]:
criterion = FocalLoss(alpha=0.25, gamma=2)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [62]:
import torch
import numpy as np
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    f1_score
)

EPS = 1e-8

In [63]:
def find_best_thresholds(y_true, y_prob):
    """
    Find per-class threshold maximizing F1
    """
    num_classes = y_true.shape[1]
    thresholds = []

    for c in range(num_classes):
        precision, recall, th = precision_recall_curve(
            y_true[:, c],
            y_prob[:, c]
        )

        f1 = 2 * precision * recall / (precision + recall + EPS)
        best_idx = np.argmax(f1)
        thresholds.append(th[min(best_idx, len(th)-1)])

    return np.array(thresholds)

In [65]:
def compute_core_metrics(y_true, y_prob, thresholds):
    preds = (y_prob >= thresholds).astype(int)

    TP = (preds * y_true).sum(axis=0)
    FP = (preds * (1 - y_true)).sum(axis=0)
    FN = ((1 - preds) * y_true).sum(axis=0)

    # ----- Per-class -----
    CP_c = TP / (TP + FP + EPS)
    CR_c = TP / (TP + FN + EPS)
    CF1_c = 2 * CP_c * CR_c / (CP_c + CR_c + EPS)

    CP = CP_c.mean()
    CR = CR_c.mean()
    CF1 = CF1_c.mean()

    # ----- Overall -----
    TP_sum = TP.sum()
    FP_sum = FP.sum()
    FN_sum = FN.sum()

    OP = TP_sum / (TP_sum + FP_sum + EPS)
    OR = TP_sum / (TP_sum + FN_sum + EPS)
    OF1 = 2 * OP * OR / (OP + OR + EPS)

    return CP, CR, CF1, OP, OR, OF1

In [66]:
def compute_ranking_metrics(y_true, y_prob):

    APs = []
    AUCs = []

    for c in range(y_true.shape[1]):
        if y_true[:, c].sum() > 0:
            APs.append(
                average_precision_score(y_true[:, c], y_prob[:, c])
            )
            AUCs.append(
                roc_auc_score(y_true[:, c], y_prob[:, c])
            )

    mAP = np.mean(APs)
    macro_auc = np.mean(AUCs)

    return mAP, macro_auc, APs, AUCs

In [1]:
def evaluate_multilabel(all_logits, all_targets):

    y_prob = torch.sigmoid(all_logits).cpu().numpy()
    y_true = all_targets.cpu().numpy()

    # ---- Threshold search ----
    thresholds = find_best_thresholds(y_true, y_prob)

    # ---- Core metrics ----
    CP, CR, CF1, OP, OR, OF1 = compute_core_metrics(
        y_true, y_prob, thresholds
    )

    # ---- Ranking metrics ----
    mAP, macro_auc, APs, AUCs = compute_ranking_metrics(
        y_true, y_prob
    )

    # ---- Additional F1 ----
    preds = (y_prob >= thresholds).astype(int)

    micro_f1 = f1_score(y_true, preds, average="micro")
    macro_f1 = f1_score(y_true, preds, average="macro")

    return {
        "mAP": mAP,
        "Macro_AUC": macro_auc,
        "OP": OP,
        "OR": OR,
        "OF1": OF1,
        "CP": CP,
        "CR": CR,
        "CF1": CF1,
        "Micro_F1": micro_f1,
        "Macro_F1": macro_f1,
        "thresholds": thresholds,
        "per_class_AP": APs,
        "per_class_AUC": AUCs
    }

In [2]:
import torch
import os

save_dir = "/kaggle/working"
best_of1 = 0.0

epochs = 100

for epoch in range(epochs):

    # ======================
    # TRAINING
    # ======================
    model.train()
    total_loss = 0

    for images, ages, labels in train_loader:

        images = images.to(device)
        ages = ages.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images, ages)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    # ======================
    # VALIDATION
    # ======================
    model.eval()

    all_logits = []
    all_targets = []

    with torch.no_grad():
        for images, ages, labels in val_loader:

            images = images.to(device)
            ages = ages.to(device)
            labels = labels.to(device)

            logits = model(images, ages)

            all_logits.append(logits.detach().cpu())
            all_targets.append(labels.detach().cpu())

    all_logits = torch.cat(all_logits)
    all_targets = torch.cat(all_targets)

    metrics = evaluate_multilabel(all_logits, all_targets)

    # ======================
    # PRINT METRICS
    # ======================
    print(f"\nEpoch [{epoch+1}/{epochs}]")
    print(f"Train Loss : {train_loss:.4f}")

    print(
        f"mAP:{metrics['mAP']:.4f} | "
        f"OP:{metrics['OP']:.4f} | "
        f"OR:{metrics['OR']:.4f} | "
        f"OF1:{metrics['OF1']:.4f}"
    )

    print(
        f"CP:{metrics['CP']:.4f} | "
        f"CR:{metrics['CR']:.4f} | "
        f"CF1:{metrics['CF1']:.4f}"
    )

    print(
        f"MacroAUC:{metrics['Macro_AUC']:.4f} | "
        f"MicroF1:{metrics['Micro_F1']:.4f}"
    )

    # ======================
    # SAVE BEST MODEL
    # ======================
    if metrics["OF1"] > best_of1:
        best_of1 = metrics["OF1"]

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "metrics": metrics
        },
        os.path.join(save_dir, "best_medfusionnet.pth"))

        print("✅ Best model saved!")

    # ======================
    # SAVE LAST CHECKPOINT
    # ======================
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics
    },
    os.path.join(save_dir, "last_medfusionnet.pth"))

NameError: name 'model' is not defined

In [20]:
import torch
from PIL import Image
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Image preprocessing (same as validation)
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

# Load image
img = Image.open("/kaggle/input/datasets/soubhagyakhan/test-images/A Khan X-Ray.png").convert("RGB")
img = transform(img).unsqueeze(0).to(device)

# Age input
age = torch.tensor([[48.0]]).to(device)

# Load model
checkpoint = torch.load("/kaggle/input/models/soubhagyakhan/best-medfustionnet/pytorch/default/1/best_medfusionnet.pth", map_location=device,weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

with torch.no_grad():

    logits = model(img, age)

    probs = torch.sigmoid(logits)

print(probs)

tensor([[0.3971, 0.0937, 0.3479, 0.2408, 0.5354, 0.1325, 0.0995, 0.0246, 0.3540,
         0.3152, 0.2597, 0.1832, 0.1440, 0.2105]], device='cuda:0')


In [ ]:
[np.str_('Atelectasis'), np.str_('Cardiomegaly'), np.str_('Consolidation'), np.str_('Edema'), np.str_('Effusion'), np.str_('Emphysema'), np.str_('Fibrosis'), np.str_('Hernia'), np.str_('Infiltration'), np.str_('Mass'), np.str_('Nodule'), np.str_('Pleural_Thickening'), np.str_('Pneumonia'), np.str_('Pneumothorax')]